# 🏗️ 03 - Attribute Engineering

## 0. Introducción ⚙️
En este notebook se realiza la ingeniería de atributos sobre el dataset de tumores cerebrales previamente limpiado. El objetivo es transformar y codificar las variables para que sean aptas para el entrenamiento de modelos de machine learning.

## 1. Importación de Librerías y Carga de Datos 📚
Se importa pandas y las tecnicas de preprocesamiento del modulo `sklearn.preprocessing`. Además se carga el dataset procesado en la etapa anterior.

In [229]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

In [230]:
df = pd.read_csv('../data/raw/brain_tumor_dataset.csv')
df.head()

,Patient_ID,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
0,1,73,Male,Malignant,5.375612,Temporal,Astrocytoma,III,Vision Issues,Seizures,Seizures,No,No,No,51.312579,0.111876,No,Positive,Yes
1,2,26,Male,Benign,4.847098,Parietal,Glioblastoma,II,Headache,Headache,Nausea,Yes,Yes,Yes,46.373273,2.165736,Yes,Positive,Yes
2,3,31,Male,Benign,5.588391,Parietal,Meningioma,I,Vision Issues,Headache,Seizures,No,No,No,47.072221,1.884228,No,Negative,No
3,4,29,Male,Malignant,1.436600,Temporal,Medulloblastoma,IV,Vision Issues,Seizures,Headache,Yes,No,Yes,51.853634,1.283342,Yes,Negative,No
4,5,54,Female,Benign,2.417506,Parietal,Glioblastoma,I,Headache,Headache,Seizures,No,No,Yes,54.708987,2.069477,No,Positive,Yes


## 2. Eliminación de Columnas Innecesarias 🚮
Identificamos las columnas innecesarias para nuestro procesamiento y el futuro entrenamiento de modelos y las eliminamos.

In [231]:
df.drop(columns=['Patient_ID'], inplace=True)

## 3. Conversión de Variables Ordinales Ⅳ → 4

La variable 'Stage' representa el estadio del tumor usando números romanos (I, II, III, IV). Para que los modelos puedan interpretar correctamente esta información ordinal, se realiza la conversión a valores numéricos enteros usando `.map()`, preservando el orden natural de los estadios.

Esto facilita el análisis y el entrenamiento de modelos que pueden beneficiarse de la información de orden en la variable.

In [232]:
roman_to_int = {'I': 1, 'II': 2, 'III': 3, 'IV': 4}
df['Stage'] = df['Stage'].map(roman_to_int)

df.head()

,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
0,73,Male,Malignant,5.375612,Temporal,Astrocytoma,3,Vision Issues,Seizures,Seizures,No,No,No,51.312579,0.111876,No,Positive,Yes
1,26,Male,Benign,4.847098,Parietal,Glioblastoma,2,Headache,Headache,Nausea,Yes,Yes,Yes,46.373273,2.165736,Yes,Positive,Yes
2,31,Male,Benign,5.588391,Parietal,Meningioma,1,Vision Issues,Headache,Seizures,No,No,No,47.072221,1.884228,No,Negative,No
3,29,Male,Malignant,1.436600,Temporal,Medulloblastoma,4,Vision Issues,Seizures,Headache,Yes,No,Yes,51.853634,1.283342,Yes,Negative,No
4,54,Female,Benign,2.417506,Parietal,Glioblastoma,1,Headache,Headache,Seizures,No,No,Yes,54.708987,2.069477,No,Positive,Yes


## 4. Definición de una Nueva Variable Objetivo 🤖

Durante la etapa de análisis exploratorio (EDA), el **heatmap de correlación** reveló que no existe una relación significativa entre la mayoría de las variables del dataset original. Esta falta de correlación afectó directamente el rendimiento de los modelos de Machine Learning entrenados inicialmente, ya que estos no lograban aprender patrones significativos y se comportaban como si predijeran al azar.

Para abordar esta limitación, se decidió **crear una nueva variable objetivo sintética denominada `Risk_Prognosis`**, la cual representa un **índice de pronóstico de riesgo** basado en fundamentos clínicos y médicos que influyen en la progresión de tumores cerebrales. Esta variable fue construida combinando lógicamente múltiples características del paciente y del tumor:

- **Tipo de tumor (`Tumor_Type`)**: se asigna mayor riesgo a tumores malignos.
- **Etapa del tumor (`Stage`)**: etapas más avanzadas aportan mayor peso.
- **Tamaño del tumor (`Tumor_Size`)** y **tasa de crecimiento (`Tumor_Growth_Rate`)**: ambas variables fueron normalizadas e incorporadas como factores de riesgo.
- **Histología (`Histology`)** y **Ubicación (`Location`)**: se asignaron pesos según la gravedad asociada a ciertos tipos de tumores o regiones cerebrales.
- **Edad del paciente (`Age`)** y **antecedentes familiares (`Family_History`)**: se integraron como factores que modifican el pronóstico.

Finalmente, se añadió un leve **ruido gaussiano** para simular variabilidad natural en los datos reales.

Esta estrategia permitió construir una variable objetivo más coherente y con una relación directa con el resto de las variables predictoras, mejorando notablemente el rendimiento de los modelos entrenados.

In [ ]:
def create_risk_prognosis_target(df):
  
    df_new = df.copy()
    
    risk_score = np.zeros(len(df))
    
    risk_score += np.where(df['Tumor_Type'] == 'Malignant', 3, 1)
    
    risk_score += df['Stage'] * 0.8
    
    tumor_size_norm = (df['Tumor_Size'] - df['Tumor_Size'].min()) / (df['Tumor_Size'].max() - df['Tumor_Size'].min())
    risk_score += tumor_size_norm * 3
    
    histology_risk = {
        'Glioblastoma': 3.5,
        'Meningioma': 1.5,
        'Astrocytoma': 2.0,
        'Oligodendroglioma': 2.2,
        'Pituitary Adenoma': 1.2
    }
   
    risk_score += df['Histology'].map(histology_risk).fillna(2.0)
    
    age_risk = np.where(df['Age'] > 65, 1.5, 
                       np.where(df['Age'] > 50, 1.0, 0.5))
    risk_score += age_risk
    
    growth_norm = (df['Tumor_Growth_Rate'] - df['Tumor_Growth_Rate'].min()) / (df['Tumor_Growth_Rate'].max() - df['Tumor_Growth_Rate'].min())
    risk_score += growth_norm * 2
    
    location_risk = {
        'Brainstem': 3.0,
        'Frontal': 1.5,
        'Temporal': 2.0,
        'Parietal': 1.8,
        'Occipital': 1.6,
        'Cerebellum': 2.2
    }
    risk_score += df['Location'].map(location_risk).fillna(1.8)
    
    risk_score += np.where(df['Family_History'] == 'Yes', 1.0, 0)
    
    np.random.seed(42)
    noise = np.random.normal(0, 0.3, len(df))
    risk_score += noise
    
    df_new['Risk_Prognosis'] = [score for score in risk_score]
    
    
    return df_new, risk_score

    

In [234]:
df, scores = create_risk_prognosis_target(df)
df.head()

,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required,Risk_Prognosis
0,73,Male,Malignant,5.375612,Temporal,Astrocytoma,3,Vision Issues,Seizures,Seizures,No,No,No,51.312579,0.111876,No,Positive,Yes,12.596891
1,26,Male,Benign,4.847098,Parietal,Glioblastoma,2,Headache,Headache,Nausea,Yes,Yes,Yes,46.373273,2.165736,Yes,Positive,Yes,12.156025
2,31,Male,Benign,5.588391,Parietal,Meningioma,1,Vision Issues,Headache,Seizures,No,No,No,47.072221,1.884228,No,Negative,No,8.631772
3,29,Male,Malignant,1.436600,Temporal,Medulloblastoma,4,Vision Issues,Seizures,Headache,Yes,No,Yes,51.853634,1.283342,Yes,Negative,No,13.268716
4,54,Female,Benign,2.417506,Parietal,Glioblastoma,1,Headache,Headache,Seizures,No,No,Yes,54.708987,2.069477,No,Positive,Yes,9.993549


## 5. Normalización de Variables Numéricas 🔢
Para estandarizar las variables numéricas, se utiliza la técnica StandardScaler, que transforma los valores de cada columna restando la media y dividiendo por la desviación estándar. Esto asegura que las características tengan media 0 y varianza 1, lo cual es fundamental para algoritmos que dependen de magnitudes.
La transformación aplicada por `StandardScaler` se define matemáticamente como:

$$
z = \frac{x - \mu}{\sigma}
$$

Donde:

- \( x \) es el valor original
- \( \mu \) es la media de la característica
- \( \sigma \) es la desviación estándar de la característica


In [235]:
num_cols = ['Age', 'Tumor_Size', 'Survival_Rate', 'Tumor_Growth_Rate', 'Risk_Prognosis']
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
df.head()


,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required,Risk_Prognosis
0,1.355253,Male,Malignant,0.050488,Temporal,Astrocytoma,3,Vision Issues,Seizures,Seizures,No,No,No,-1.089675,-1.717548,No,Positive,Yes,0.381559
1,-1.347627,Male,Benign,-0.141399,Parietal,Glioblastoma,2,Headache,Headache,Nausea,Yes,Yes,Yes,-1.375673,0.739298,Yes,Positive,Yes,0.160638
2,-1.060087,Male,Benign,0.127742,Parietal,Meningioma,1,Vision Issues,Headache,Seizures,No,No,No,-1.335202,0.402556,No,Negative,No,-1.605393
3,-1.175103,Male,Malignant,-1.379648,Temporal,Medulloblastoma,4,Vision Issues,Seizures,Headache,Yes,No,Yes,-1.058346,-0.316229,Yes,Negative,No,0.718216
4,0.262599,Female,Benign,-1.023511,Parietal,Glioblastoma,1,Headache,Headache,Seizures,No,No,Yes,-0.893014,0.624153,No,Positive,Yes,-0.922996


## 6. Codificación de Variables Categóricas 🏷️

Las variables categóricas nominales binarias se transforman mediante **Label Encoding**, asignando un valor numérico (0 o 1) a cada categoría. Esta técnica es eficiente para columnas con solo dos categorías, como 'Yes' / 'No', ya que evita la creación de múltiples columnas como en el One Hot Encoding.

Se utiliza `LabelEncoder` de `sklearn.preprocessing`, lo cual facilita la conversión de datos categóricos al formato requerido por algoritmos de aprendizaje automático.


In [236]:
le = LabelEncoder()
le_cols = ['Gender', 'Tumor_Type', 'Family_History', 'MRI_Result',
        'Radiation_Treatment', 'Surgery_Performed', 'Chemotherapy', 'Follow_Up_Required']

for col in le_cols:
    df[col] = le.fit_transform(df[col])

df.head()

,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required,Risk_Prognosis
0,1.355253,1,1,0.050488,Temporal,Astrocytoma,3,Vision Issues,Seizures,Seizures,0,0,0,-1.089675,-1.717548,0,1,1,0.381559
1,-1.347627,1,0,-0.141399,Parietal,Glioblastoma,2,Headache,Headache,Nausea,1,1,1,-1.375673,0.739298,1,1,1,0.160638
2,-1.060087,1,0,0.127742,Parietal,Meningioma,1,Vision Issues,Headache,Seizures,0,0,0,-1.335202,0.402556,0,0,0,-1.605393
3,-1.175103,1,1,-1.379648,Temporal,Medulloblastoma,4,Vision Issues,Seizures,Headache,1,0,1,-1.058346,-0.316229,1,0,0,0.718216
4,0.262599,0,0,-1.023511,Parietal,Glioblastoma,1,Headache,Headache,Seizures,0,0,1,-0.893014,0.624153,0,1,1,-0.922996


## 7. Codificación de Variables Categóricas Nominales con One Hot Encoding 0️⃣1️⃣

Para las variables categóricas con múltiples categorías posibles, como la localización del tumor, el tipo histológico y los síntomas, se utiliza **One Hot Encoding**. Esta técnica crea una columna binaria para cada categoría, permitiendo que los modelos interpreten correctamente la información sin asumir un orden implícito entre las categorías.

Se emplea la función `pd.get_dummies` de pandas, que genera automáticamente las columnas necesarias y asigna valores 0 o 1 según corresponda.

In [237]:
onehot_cols = ['Location', 'Histology', 'Symptom_1', 'Symptom_2', 'Symptom_3']
df = pd.get_dummies(df, columns=onehot_cols, drop_first=True, dtype=int)

df.head()

,Age,Gender,Tumor_Type,Tumor_Size,Stage,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,...,Histology_Meningioma,Symptom_1_Nausea,Symptom_1_Seizures,Symptom_1_Vision Issues,Symptom_2_Nausea,Symptom_2_Seizures,Symptom_2_Vision Issues,Symptom_3_Nausea,Symptom_3_Seizures,Symptom_3_Vision Issues
0,1.355253,1,1,0.050488,3,0,0,0,-1.089675,-1.717548,...,0,0,0,1,0,1,0,0,1,0
1,-1.347627,1,0,-0.141399,2,1,1,1,-1.375673,0.739298,...,0,0,0,0,0,0,0,1,0,0
2,-1.060087,1,0,0.127742,1,0,0,0,-1.335202,0.402556,...,1,0,0,1,0,0,0,0,1,0
3,-1.175103,1,1,-1.379648,4,1,0,1,-1.058346,-0.316229,...,0,0,0,1,0,1,0,0,0,0
4,0.262599,0,0,-1.023511,1,0,0,1,-0.893014,0.624153,...,0,0,0,0,0,0,0,0,1,0


## 8. Exportación del Dataset 📦 

In [238]:
import os

os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/brain_tumor_preprocessed.csv", index=False)